In [1]:
from laya import Router
import pandas as pd
import json

router = Router()  # downloads a checkpoint on first use; Router(preload=True) loads all three up front

In [2]:
questions = {
    "measurement_type": {"type": "choice", "instructions": "What type of measurement does this appear to be?",
                   "criteria": {"pH": "The pH of the broth. Represented as a list of numbers that MUST be between 0 and 14, value of each item is close to the previous. Typical values are between 4 and 8. Often named a variant of 'PH'.",
                                "Temperature": "The temperature of the broth. Represented as a list of numbers that are usually between 20 and 40, but sometimes goes out of that range. Often named 'T', 'Temp', or some variation of that",
                                "Stirrate": "The RPM of the impeller in the broth. Represented as a list of numbers which are typically greater than 100. Value of each item may be very different from the previous. Values tend to be higher than or the same as the previous value, Often labeled 'Stirr' or 'Agit'",
                                "Other": "Any measurement that doesn't appear to be pH, Temperature, or Stirrate"}},
}

In [3]:
def predict_canonical_label(column: pd.Series, questions = questions, correct_answer = None, confidence_threshold=0.5, meta_only=False) -> float:  
    meta = getattr(column,'meta') if hasattr(column,'meta') else None
    if meta_only:
        if not meta and not column.name:
            print('⚠️ Warning: Using meta-only, but all metadata is null')
        meta = meta or {} # Handling the case where the only metadata is the name
        state = json.dumps({'name': column.name, **meta})
    elif meta:
        state = json.dumps({column.name: {'data':list(column), 'meta':meta}})
    else:
        state = json.dumps({column.name: list(column)})
    
    result = router.predict(state, questions)

    #Pretty Print Logic
    print('---')
    print(f'➡️ {result["answers"]["measurement_type"]["choice"]}')
    print('\n')
    print(f'🤔 Answer Confidence: {result["answers"]["measurement_type"]["answer_confidence"]}')
    print(f'📊 Probabilities: {result["answers"]["measurement_type"]["probabilities"]}')
    if correct_answer:
        print('\n')
        if correct_answer == result["answers"]["measurement_type"]["choice"]:
            print(f'✅ Choice is correct: {result["answers"]["measurement_type"]["choice"]} = {correct_answer}')
        else:
            print(f'❌ Choice is wrong: {result["answers"]["measurement_type"]["choice"]} != {correct_answer}')
        if result["answers"]["measurement_type"]["answer_confidence"] < confidence_threshold:
            print(f'🤷 But confidence is below threshold of {confidence_threshold}')
    print('---')

    return result["answers"]["measurement_type"]["answer_confidence"]

### Labeled data with just a few data points
#### pH

In [5]:
column = pd.Series([5.4,5.6,5.5,5.5],name='ph')
confidence = predict_canonical_label(column, correct_answer = 'pH')

---
➡️ pH


🤔 Answer Confidence: 0.8048
📊 Probabilities: {'pH': 0.8048, 'Temperature': 0.037, 'Stirrate': 0.0325, 'Other': 0.1257}


✅ Choice is correct: pH = pH
---


This is accurate, but the confidence is pretty low, considering that the name is literally 'ph'.

### Temperature

In [6]:
column = pd.Series([30.0,30.9,31.5,29.8], name='Temp')
confidence = predict_canonical_label(column, correct_answer = 'Temperature')

#Let's test another common label
print("\nCase with just 'T' as the label")
column = pd.Series([30.0,30.9,31.5,29.8], name='T')
confidence = predict_canonical_label(column, correct_answer = 'Temperature')

---
➡️ Temperature


🤔 Answer Confidence: 0.9643
📊 Probabilities: {'pH': 0.0033, 'Temperature': 0.9643, 'Stirrate': 0.0048, 'Other': 0.0276}


✅ Choice is correct: Temperature = Temperature
---

Case with just 'T' as the label
---
➡️ Other


🤔 Answer Confidence: 0.764
📊 Probabilities: {'pH': 0.0982, 'Temperature': 0.0537, 'Stirrate': 0.0841, 'Other': 0.764}


❌ Choice is wrong: Other != Temperature
---


Still looks good. Maybe worth noting that the less descriptive 'T' case is much less confident. A tweaked confidence threshold would pass that for manual labelling.

#### Stirrate

In [7]:
column = pd.Series([200,300,350,400,425], name='STIRR')
confidence = predict_canonical_label(column, correct_answer = 'Stirrate')

---
➡️ Stirrate


🤔 Answer Confidence: 0.5135
📊 Probabilities: {'pH': 0.0879, 'Temperature': 0.0921, 'Stirrate': 0.5135, 'Other': 0.3065}


✅ Choice is correct: Stirrate = Stirrate
---


### Other

In [8]:
column = pd.Series([1,0,0,1,0], name='Pump1')
confidence = predict_canonical_label(column, correct_answer = 'Other')

---
➡️ Other


🤔 Answer Confidence: 0.6688
📊 Probabilities: {'pH': 0.1271, 'Temperature': 0.0848, 'Stirrate': 0.1193, 'Other': 0.6688}


✅ Choice is correct: Other = Other
---


Honestly, for the compute cost this already useful - even if it does require us naming something. It's waaay worse than looking up in a hash table or using a regular expression for pattern matching, but not having to write a bunch of patterns or examples saves enough development time that I'd happily take it.

### Unlabeled data

In [9]:
column = pd.Series([5.4,5.6,5.5,5.5])
confidence = predict_canonical_label(column, correct_answer = 'pH')

---
➡️ Other


🤔 Answer Confidence: 0.5519
📊 Probabilities: {'pH': 0.1796, 'Temperature': 0.1523, 'Stirrate': 0.1162, 'Other': 0.5519}


❌ Choice is wrong: Other != pH
---


It seems like we've lost a lot of predictive power when we just look at the numbers and don't provide any metadata. To be fair, as a domain expert I might suggest this could be pH, but definitely wouldn't be confident that it's not 'Other', since Other is defined so broadly.

I note that pH is the second-highest confidence guess. So it's possible we could tweak the options to get better one-shot answers, or imagine some kind of loop that refines the guess if the first-pass is 'Other'. Let's explore more before going down those roads.

#### With temperature

In [10]:
column = pd.Series([30.0,30.9,31.5,29.8])
confidence = predict_canonical_label(column, correct_answer = 'Temperature')

---
➡️ Other


🤔 Answer Confidence: 0.6842
📊 Probabilities: {'pH': 0.1297, 'Temperature': 0.112, 'Stirrate': 0.0741, 'Other': 0.6842}


❌ Choice is wrong: Other != Temperature
---


Again, 'Other' is acting as a kind of attractor. We really don't have much information from these five datapoints! Troublingly, though, the confidence for pH and Temperature are about the same, which should be impossible given my instructions that pH can't be greater than 14. A domain expert would likely say the chances of this being pH are infinitessimally small.

Let's poke at this for a moment - I also indicated that for pH the 'value of each item is close to the previous'. What if we varied the numbers greatly item to item?

In [11]:
column = pd.Series([30.0,38.9,20.5,29.8])
confidence = predict_canonical_label(column, correct_answer = 'Temperature')

---
➡️ Other


🤔 Answer Confidence: 0.6858
📊 Probabilities: {'pH': 0.125, 'Temperature': 0.1019, 'Stirrate': 0.0873, 'Other': 0.6858}


❌ Choice is wrong: Other != Temperature
---


Nope! This actually made pH apparently *more* likely. So the idea of just reprocessing anything labeled 'Other' doesn't seem helpful since our first pass really isn't helping us refine anything in the first place.

### Unlabeled data, but there's a lot of it

Most datastreams in fermentation won't be five items long. Is it possible that by creating a much longer stream we can increase our confidence?

In [12]:
import random # rather than handcrafting a stream, I'll just make a random one

In [14]:
column = pd.Series([5.5 + random.random() - 0.5 for x in range(500)]) #1000 points is too long for the context window. Going with 500
confidence = predict_canonical_label(column, correct_answer = 'pH')

large_ph_column = column #saving this for reusing later

---
➡️ Other


🤔 Answer Confidence: 0.3629
📊 Probabilities: {'pH': 0.2717, 'Temperature': 0.1944, 'Stirrate': 0.1711, 'Other': 0.3629}


❌ Choice is wrong: Other != pH
🤷 But confidence is below threshold of 0.5
---


This is, for all intents and purposes, a wash. Everything looks about the same and the confidence is low. We might get different results by re-running with a different random stream, but I don't have condifence it will change greatly except as statistical anomaly.

#### With Temperature

In [15]:
column = pd.Series([30 + ((random.random() - 0.5)*20) for x in range(500)])
confidence = predict_canonical_label(column, correct_answer = 'Temperature')

---
➡️ pH


🤔 Answer Confidence: 0.3224
📊 Probabilities: {'pH': 0.3224, 'Temperature': 0.2554, 'Stirrate': 0.1902, 'Other': 0.232}


❌ Choice is wrong: pH != Temperature
🤷 But confidence is below threshold of 0.5
---


Effectively the same result, except with the added insult of pH having the highest probability - the one result that is expressively forbidden by the plain-language rules.

### Unlabeled data with additional context

Unlabeled data is not looking good, but maybe we can give some extra context in the prompt itself that will help it. *I* know that this if fermentation data, so let's add that information to the question.

In [16]:
questions_with_context = {
    "measurement_type": {"type": "choice", "instructions": "Given that this is fermentation data, What type of measurement does this appear to be?",
                   "criteria": {"pH": "The pH of the broth. Represented as a list of numbers that MUST be between 0 and 14, value of each item is close to the previous. Typical values are between 4 and 8. Often named a variant of 'PH'.",
                                "Temperature": "The temperature of the broth. Represented as a list of numbers that are usually between 20 and 40, but sometimes goes out of that range. Often named 'T', 'Temp', or some variation of that",
                                "Stirrate": "The RPM of the impeller in the broth. Represented as a list of numbers which are typically greater than 100. Value of each item may be very different from the previous. Values tend to be higher than or the same as the previous value, Often labeled 'Stirr' or 'Agit'",
                                "Other": "Any measurement that doesn't appear to be pH, Temperature, or Stirrate"}},
}

#### With large dataset

In [17]:
column = large_ph_column # using the same random list as above YMMV if you run in a different order
confidence = predict_canonical_label(column, questions = questions_with_context, correct_answer = 'pH')

---
➡️ Other


🤔 Answer Confidence: 0.3721
📊 Probabilities: {'pH': 0.2858, 'Temperature': 0.1858, 'Stirrate': 0.1563, 'Other': 0.3721}


❌ Choice is wrong: Other != pH
🤷 But confidence is below threshold of 0.5
---


Doesn't look like it made a difference.

#### With short dataset

In [18]:
column = pd.Series([5.4,5.6,5.5,5.5])
confidence = predict_canonical_label(column, questions = questions_with_context, correct_answer = 'pH')

---
➡️ Other


🤔 Answer Confidence: 0.4362
📊 Probabilities: {'pH': 0.2259, 'Temperature': 0.1986, 'Stirrate': 0.1392, 'Other': 0.4362}


❌ Choice is wrong: Other != pH
🤷 But confidence is below threshold of 0.5
---


Once again, wrong and the non-Other choices are fairly close in likelihood, so this doesn't tell us anything.

It's possible that we would get better results if we added yet more context, or re-phrased the options very carefully, but I don't want to go down the rabbit hole of prompt engineering, so I'll just leave it there for now. The goal of this experiment is to see if we can get the low-hanging fruit with minimal effort, and carefully wording the prompts sounds to me like more work than just manually labelling or creating strict numerical rules for most datasets.

### Providing Units

So far we have no evidence that we can assess the data very well, but we did see good results when there was some verbal metadata associated with the dataset - we even got 'T' normalized to 'Temperature'. Units of measure are a very strong indicator for many datastreams - let's add a some metadata into the state and see if we get better results.

#### With Temperature

In [19]:
column = pd.Series([30.0,30.9,31.5,29.8])
column.meta = {"units":"degC"}
confidence = predict_canonical_label(column, correct_answer = 'Temperature')

---
➡️ Temperature


🤔 Answer Confidence: 0.6028
📊 Probabilities: {'pH': 0.0608, 'Temperature': 0.6028, 'Stirrate': 0.0524, 'Other': 0.284}


✅ Choice is correct: Temperature = Temperature
---


I didn't describe units of measure in the question criteria, but including some made a big difference, even without any other label. I suspect we're benefitting from something trained into the model about the concept of 'Temperature'. So that's nice - free context!

#### With Stirrate

In [20]:
column = pd.Series([30.0,30.9,31.5,29.8]) #Note we didn't actually change the input data!
column.meta = {"units":"rpm"}
confidence = predict_canonical_label(column, correct_answer = 'Stirrate')

column = pd.Series([300,409,515,598]) #Changing the data, in fact, had almost no effect on the choice
column.meta = {"units":"rpm"}
confidence = predict_canonical_label(column, correct_answer = 'Stirrate')

---
➡️ Stirrate


🤔 Answer Confidence: 0.4409
📊 Probabilities: {'pH': 0.1092, 'Temperature': 0.0996, 'Stirrate': 0.4409, 'Other': 0.3503}


✅ Choice is correct: Stirrate = Stirrate
🤷 But confidence is below threshold of 0.5
---
---
➡️ Stirrate


🤔 Answer Confidence: 0.4426
📊 Probabilities: {'pH': 0.1112, 'Temperature': 0.1086, 'Stirrate': 0.4426, 'Other': 0.3377}


✅ Choice is correct: Stirrate = Stirrate
🤷 But confidence is below threshold of 0.5
---


The guess is right, even though the confidence is low. Looking at the probabilities, I at least feel like it's fair to say 'Other' is a possibility. For instance, pump speed can also be measured in RPM.

Probably more important to note is that the actual data didn't seem to affect the assessment at all. The data that was previously assessed as Temperature was assessed as Stirrate when we changed the units to 'rpm'. *Then*, when we changed the data to look more 'Stirrate-like', the assessment remained about the same.

Are we doing anything besides injecting noise with these datastreams? What if we didn't try to assess on the data at all?

#### Assessing only on metadata

In [21]:
column = pd.Series([300,409,515,598])
column.meta = {"units":"rpm"}

confidence = predict_canonical_label(column, correct_answer = 'Stirrate', meta_only = True) #model won't even look at the data

---
➡️ Stirrate


🤔 Answer Confidence: 0.7375
📊 Probabilities: {'pH': 0.0916, 'Temperature': 0.0789, 'Stirrate': 0.7375, 'Other': 0.092}


✅ Choice is correct: Stirrate = Stirrate
---


Look at that - it's notably more confident about Stirrate. The numbers don't seem like they were helping us at all!

#### Purposefully getting 'Other'

Since the units seem like a strong indicator, can we affirmatively get an 'Other' result by using units that don't fit any of our options?

In [22]:
column = pd.Series([30.0,30.9,31.5,29.8])
column.meta = {"units":"mmol/l/hr"}
confidence = predict_canonical_label(column, correct_answer = 'Other')
confidence = predict_canonical_label(column, correct_answer = 'Other',meta_only=True)

---
➡️ Other


🤔 Answer Confidence: 0.5333
📊 Probabilities: {'pH': 0.266, 'Temperature': 0.1233, 'Stirrate': 0.0773, 'Other': 0.5333}


✅ Choice is correct: Other = Other
---
---
➡️ Other


🤔 Answer Confidence: 0.39
📊 Probabilities: {'pH': 0.2903, 'Temperature': 0.1842, 'Stirrate': 0.1355, 'Other': 0.39}


✅ Choice is correct: Other = Other
🤷 But confidence is below threshold of 0.5
---


Well, not bad. This time we saw that only submitting the metadata hurt our confidence, but we did guess correctly in both cases. I'm not sure if we can say the data is doing nothing - it might be having an effect, or it might be coincidentally sometimes adding to the system in a such a way that we get nudged toward the right guess. Either way, we definitely don't have proof that we can rely on it.

### Pre-calculating Summary Statistics

Let's further test the idea that metadata can help us more than the data itself. What if we aggregated the dataset using traditional methods and sent the model that aggregation instead of the data. In theory, this are fewer tokens containing denser information because we've pre-calculated things like range and mean.

In [23]:
column = pd.Series([30.0,38.9,20.5,29.8])

column.meta = dict(column.describe()) #Dumping the generic summary into the metadata
# Looks like: '{"count": 4.0, "mean": 29.8, "std": 7.513099671019057, "min": 20.5, "25%": 27.475, "50%": 29.9, "75%": 32.225, "max": 38.9}'

confidence = predict_canonical_label(column, correct_answer = 'Temperature')

---
➡️ Other


🤔 Answer Confidence: 0.4718
📊 Probabilities: {'pH': 0.2115, 'Temperature': 0.152, 'Stirrate': 0.1646, 'Other': 0.4718}


❌ Choice is wrong: Other != Temperature
🤷 But confidence is below threshold of 0.5
---


That doesn't seem to have helped. For comparison, the version without these pre-calculated values had the following probabilities: `{'pH': 0.1297, 'Temperature': 0.112, 'Stirrate': 0.0741, 'Other': 0.6842}`. At least this version said it wasn't confident, but I'm not willing to ascribe that to anything we did on purpose.

At this point, I'm more inclined to believe that assessing on numbers in any form ins't very helpful. Basically, the model can't do math. Makes sense - it's almost a cliche how bad models are at math!

### Providing Minimal Context

In [24]:
minimal_context_questions = {
    "measurement_type": {"type": "choice", "instructions": "What type of measurement does this appear to be?",
                   "criteria": {"pH": "Acts like pH in a bioreactor",
                                "Temperature": "Acts like Temperature in a bioreactor",
                                "Stirrate": "Acts like stirrate in bioreactor",
                                "Other": "everything else"}},
}

In [25]:
column = pd.Series([30.0,30.9,31.5,29.8])
print('===Original Questions===')
confidence = predict_canonical_label(column, correct_answer='Temperature')
print('\n===Minimal Context Questions===')
confidence = predict_canonical_label(column,questions=minimal_context_questions, correct_answer='Temperature')

===Original Questions===
---
➡️ Other


🤔 Answer Confidence: 0.6842
📊 Probabilities: {'pH': 0.1297, 'Temperature': 0.112, 'Stirrate': 0.0741, 'Other': 0.6842}


❌ Choice is wrong: Other != Temperature
---

===Minimal Context Questions===
---
➡️ pH


🤔 Answer Confidence: 0.6747
📊 Probabilities: {'pH': 0.6747, 'Temperature': 0.1015, 'Stirrate': 0.0948, 'Other': 0.1291}


❌ Choice is wrong: pH != Temperature
---


This is actually kind of interesting - both cases are wrong, but when we don't provide any context other than "Acts like pH" and rely on the model weights to understand what that means, we get a huge swing from 'Other' to 'pH' in the prediction. This suggests two things to me:
1. There is something going here besides pure noise, but
2. The outcome is sensitive to the question we provide and how we phrase it. Possibly moreso than to the data.

In fact, let's just feed our original prompt random garbage and see what we get

#### Feeding the model random garbage

In [26]:
states = []
results = []
for i in range(1,10):
    state = ''
    for j in range(25):
        state += chr(random.randint(33,120))
    states.append(state)
    results.append(router.predict(state,questions))

In [27]:
for i, result in enumerate(results):
    print(f"{states[i]:<30} | {result['answers']['measurement_type']['choice']:<10} | {result['answers']['measurement_type']['probabilities']}")

2$_5Z\n\Y8sm4S=WrT*x6`G'B      | pH         | {'pH': 0.3631, 'Temperature': 0.1704, 'Stirrate': 0.1235, 'Other': 0.343}
303tcVQbp&Xe7Q.x6h]Yp#n#'      | pH         | {'pH': 0.3785, 'Temperature': 0.1616, 'Stirrate': 0.1336, 'Other': 0.3263}
N%(N`1^nd#+o?vS&_2bH(q6$O      | pH         | {'pH': 0.3649, 'Temperature': 0.1912, 'Stirrate': 0.1704, 'Other': 0.2735}
]Tfp1FDR'eF/$tWl/)`NR#>u>      | Other      | {'pH': 0.3336, 'Temperature': 0.1707, 'Stirrate': 0.1248, 'Other': 0.3709}
/x404,"gig9-mIqAFBP:1Hb/h      | pH         | {'pH': 0.4016, 'Temperature': 0.1752, 'Stirrate': 0.1494, 'Other': 0.2738}
&w\1rl-=EIYd'B58EZs#]vnN@      | pH         | {'pH': 0.4103, 'Temperature': 0.1572, 'Stirrate': 0.1276, 'Other': 0.3049}
@&QO;nSdYD5k&?HfrT8d_CvVc      | pH         | {'pH': 0.4361, 'Temperature': 0.1408, 'Stirrate': 0.1206, 'Other': 0.3024}
l5#TUR*a8AANG6NnOahR`&xdn      | Other      | {'pH': 0.3518, 'Temperature': 0.1593, 'Stirrate': 0.1205, 'Other': 0.3684}
Eb36^8"&rOQIfkU2N<W943pUN      | 

Well that's troubling. Ideally if we gave it random garbage it would usually mark it as 'Other', but something about the prompt seems to bias us towards 'pH'. At least the confidence value is generally low. Let's contrast with the minimal context prompt to see if we get a wildly different answer.

In [28]:
results = []
for i in range(0,9):
    results.append(router.predict(states[i],minimal_context_questions))

In [29]:
for i, result in enumerate(results):
    print(f"{states[i]:<30} | {result['answers']['measurement_type']['choice']:<10} | {result['answers']['measurement_type']['probabilities']}")

2$_5Z\n\Y8sm4S=WrT*x6`G'B      | Other      | {'pH': 0.2823, 'Temperature': 0.1485, 'Stirrate': 0.2246, 'Other': 0.3446}
303tcVQbp&Xe7Q.x6h]Yp#n#'      | pH         | {'pH': 0.3335, 'Temperature': 0.2041, 'Stirrate': 0.1633, 'Other': 0.299}
N%(N`1^nd#+o?vS&_2bH(q6$O      | Other      | {'pH': 0.1313, 'Temperature': 0.0775, 'Stirrate': 0.2076, 'Other': 0.5836}
]Tfp1FDR'eF/$tWl/)`NR#>u>      | Other      | {'pH': 0.234, 'Temperature': 0.1691, 'Stirrate': 0.2261, 'Other': 0.3708}
/x404,"gig9-mIqAFBP:1Hb/h      | Other      | {'pH': 0.2388, 'Temperature': 0.1622, 'Stirrate': 0.2107, 'Other': 0.3883}
&w\1rl-=EIYd'B58EZs#]vnN@      | Other      | {'pH': 0.2444, 'Temperature': 0.2001, 'Stirrate': 0.1688, 'Other': 0.3867}
@&QO;nSdYD5k&?HfrT8d_CvVc      | pH         | {'pH': 0.3338, 'Temperature': 0.1866, 'Stirrate': 0.1612, 'Other': 0.3184}
l5#TUR*a8AANG6NnOahR`&xdn      | Other      | {'pH': 0.2139, 'Temperature': 0.1958, 'Stirrate': 0.1608, 'Other': 0.4295}
Eb36^8"&rOQIfkU2N<W943pUN      | O

Maybe it's unsurprising that the prompt matters, but regardless, it's a problem that describing the data in inuitive language seems to have some bias.

Let's go all the way back to our inital case and see if the minimal context prompt is better or worse on an imprecisely-labeled datastream.

### Minimal context prompt with a label

In [30]:
column = pd.Series([30.0,30.9,31.5,29.8], name='Temp')
print('===Original Questions===')
confidence = predict_canonical_label(column, correct_answer='Temperature')
print('\n===Minimal Context Questions===')
confidence = predict_canonical_label(column,questions=minimal_context_questions, correct_answer='Temperature')

===Original Questions===
---
➡️ Temperature


🤔 Answer Confidence: 0.9643
📊 Probabilities: {'pH': 0.0033, 'Temperature': 0.9643, 'Stirrate': 0.0048, 'Other': 0.0276}


✅ Choice is correct: Temperature = Temperature
---

===Minimal Context Questions===
---
➡️ Temperature


🤔 Answer Confidence: 0.9986
📊 Probabilities: {'pH': 0.0004, 'Temperature': 0.9986, 'Stirrate': 0.0005, 'Other': 0.0005}


✅ Choice is correct: Temperature = Temperature
---


We actually increased our confidence. Another piece of evidence that this is not a good model for talking about numbers. Not only does numerical data not seem to help, describing the data in the prompt doesn't seem particularly helpful.

#### With pH

In [31]:
column = pd.Series([5.4,5.6,5.5,5.5],name='ph')
print('===Original Questions===')
confidence = predict_canonical_label(column, correct_answer='pH')
print('\n===Minimal Context Questions===')
confidence = predict_canonical_label(column,questions=minimal_context_questions, correct_answer='pH')

===Original Questions===
---
➡️ pH


🤔 Answer Confidence: 0.8048
📊 Probabilities: {'pH': 0.8048, 'Temperature': 0.037, 'Stirrate': 0.0325, 'Other': 0.1257}


✅ Choice is correct: pH = pH
---

===Minimal Context Questions===
---
➡️ pH


🤔 Answer Confidence: 0.9994
📊 Probabilities: {'pH': 0.9994, 'Temperature': 0.0001, 'Stirrate': 0.0001, 'Other': 0.0003}


✅ Choice is correct: pH = pH
---


Again, improved confidence. Even more this time.

#### With Stirrate

In [32]:
column = pd.Series([200,300,350,400,425], name='STIRR')
print('===Original Questions===')
confidence = predict_canonical_label(column, correct_answer='Stirrate')
print('\n===Minimal Context Questions===')
confidence = predict_canonical_label(column,questions=minimal_context_questions, correct_answer='Stirrate')

===Original Questions===
---
➡️ Stirrate


🤔 Answer Confidence: 0.5135
📊 Probabilities: {'pH': 0.0879, 'Temperature': 0.0921, 'Stirrate': 0.5135, 'Other': 0.3065}


✅ Choice is correct: Stirrate = Stirrate
---

===Minimal Context Questions===
---
➡️ Stirrate


🤔 Answer Confidence: 0.9793
📊 Probabilities: {'pH': 0.0064, 'Temperature': 0.0091, 'Stirrate': 0.9793, 'Other': 0.0052}


✅ Choice is correct: Stirrate = Stirrate
---


Huge improvement! Starting to feel like dealing with numbers on both the prompt and data side was just a way to inject noise.

#### With 'Other'

In [33]:
column = pd.Series([1,0,0,1,0], name='Pump1')
print('===Original Questions===')
confidence = predict_canonical_label(column, correct_answer='Other')
print('\n===Minimal Context Questions===')
confidence = predict_canonical_label(column,questions=minimal_context_questions, correct_answer='Other')

===Original Questions===
---
➡️ Other


🤔 Answer Confidence: 0.6688
📊 Probabilities: {'pH': 0.1271, 'Temperature': 0.0848, 'Stirrate': 0.1193, 'Other': 0.6688}


✅ Choice is correct: Other = Other
---

===Minimal Context Questions===
---
➡️ Other


🤔 Answer Confidence: 0.6888
📊 Probabilities: {'pH': 0.1293, 'Temperature': 0.0465, 'Stirrate': 0.1354, 'Other': 0.6888}


✅ Choice is correct: Other = Other
---


Weirdly, got the same exact confidence level here, but in general this is in-line with expectations.

All in all, this is a strong indicator to me that the original dream is still alive, just not the way I thought. If I want a very fast first-pass labelling system, I can just rely on the model training and get decent zero-shot responses as long as I have some amount of metadata. In fact, I shouldn't bother describing anything about the data if I want to do it this way. It was meant to be a quick and first approach, and while it's probably a little dirtier in practice, it's also even quicker than I was hoping.

### Repeatability

Since we're seeing that small changes can nudge the assessment into different outcomes, let's double-check that the same input always gives the same output in this model's architecture. For this kind of labelling we probably have a little tolerance for noise, but it would be best if outputs were deterministic if not always perfect.

In [34]:
import numpy as np

In [35]:
np.mean([1,3])

np.float64(2.0)

In [ ]:
column = pd.Series([5.4,5.6,5.5,5.5])

confidences = list()

for rep in range(100):
    confidences.append(predict_canonical_label(column))

In [37]:
print(np.mean(confidences))
print(np.std(confidences))

0.5519000000000001
1.1102230246251565e-16


✅ Well, there you go! Glad to see we're getting deterministic, if chaotic, output

## Stepping Back: How Much Math Can The Model Handle?

We've inferred from playing around here that numerical reasoning are - as expected - not the model's strong suit. Still, Let's test that more explicitly.

In [38]:
math_questions = {
    "is_one": {"type": "noul", "instructions": "Does this represent the number one?"},
    "positive": {"type": "noul", "instructions": "Does this represent a positive number?"},
    "greater_than_five": {"type": "noul", "instructions": "Does this represent a number greater than five?",
}}

num2words_dict = {-20: 'minus twenty', -19: 'minus nineteen', -18: 'minus eighteen', -17: 'minus seventeen', -16: 'minus sixteen', -15: 'minus fifteen', -14: 'minus fourteen', -13: 'minus thirteen', -12: 'minus twelve', -11: 'minus eleven', -10: 'minus ten', -9: 'minus nine', -8: 'minus eight', -7: 'minus seven', -6: 'minus six', -5: 'minus five', -4: 'minus four', -3: 'minus three', -2: 'minus two', -1: 'minus one', 0: 'zero', 1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine', 10: 'ten', 11: 'eleven', 12: 'twelve', 13: 'thirteen', 14: 'fourteen', 15: 'fifteen', 16: 'sixteen', 17: 'seventeen', 18: 'eighteen', 19: 'nineteen'}


In [41]:
print(f"{f'    |   =1   |   >0   |   >5  ':<35} ||| {f'    |   =1   |   >0   |   >5  ':>55}")

for i in range(-20,20):
    result = router.predict(str(i), math_questions)
    text_result = router.predict(num2words_dict[i], math_questions)

    number_result_output = f"{i:>3} | {result['answers']['is_one']['noul']:<6} | {result['answers']['positive']['noul']:<6} | {result['answers']['greater_than_five']['noul']:<6}"
    text_result_output = f"As Text (\"{num2words_dict[i]}\") | {text_result['answers']['is_one']['noul']:<6} | {text_result['answers']['positive']['noul']:<6} | {text_result['answers']['greater_than_five']['noul']:<6}"
    print(f"{number_result_output:<35} ||| {text_result_output:>55}")

    |   =1   |   >0   |   >5        |||                              |   =1   |   >0   |   >5  
-20 | 0.0391 | 0.001  | 0.9413      |||     As Text ("minus twenty") | 0.0544 | 0.0    | 0.0904
-19 | 0.0249 | 0.0    | 0.7         |||   As Text ("minus nineteen") | 0.0304 | 0.0    | 0.0323
-18 | 0.0238 | 0.0    | 0.9517      |||   As Text ("minus eighteen") | 0.0032 | 0.0    | 0.0004
-17 | 0.036  | 0.0001 | 0.9073      |||  As Text ("minus seventeen") | 0.0276 | 0.0001 | 0.1297
-16 | 0.0191 | 0.0019 | 0.9502      |||    As Text ("minus sixteen") | 0.0252 | 0.0    | 0.3313
-15 | 0.0273 | 0.0003 | 0.8897      |||    As Text ("minus fifteen") | 0.0413 | 0.0001 | 0.1124
-14 | 0.0222 | 0.0003 | 0.8499      |||   As Text ("minus fourteen") | 0.0283 | 0.0    | 0.0015
-13 | 0.029  | 0.0001 | 0.6065      |||   As Text ("minus thirteen") | 0.0251 | 0.0    | 0.0529
-12 | 0.0133 | 0.0    | 0.8968      |||     As Text ("minus twelve") | 0.0038 | 0.0    | 0.041 
-11 | 0.0436 | 0.0    | 0.8551      ||| 

Well, that's kind a mess. I see some specific ways the model is tripped up ("-20" looks an awful lot like "20" if you're reading the characters as text, so I see why you might consider it positive), and some ways that don't make any sense to me (Why is "two" the most likely text to represent the value one? More than "one"??).

It's pretty easy to say that this approach is likely not the fastest path to numerical assessment. Still, I think a lot of domain experts would use numerical fuzzy logic language when describing how they assess datastreams, so I don't think we should give up on this totally.

## Takeaways

* It's pretty helpful that with a very quick, plain-text description you're able to get _some_ degree of accuracy in labelling
* You can run this perfectly well on a regular old laptop!
* Numerical rules aren't turnkey. I still feel like there must be a way to integrate it in, but it's possible that eventually you're just not using the model anymore.
* All of this was done without any fine-tuning, so it's value here is that it can handle the low-hanging fruit with really minimal work. Fine-tuning could make it more effective.
* Without structured fine-tuning, it's pretty easy to get lost down the rabbit hole of refining your inputs or description, but this is imprecise work and I think, wihout a holistic understanding of how the model assesses, it will waste more time than it will save.

I think this could be really helpful if you're doing a first pass on cleaning up a raw dataset - especially if you didn't generate it and can't talk to the person who did.

It might also have some value in a larger data management system to do auto-labelling, although, again, without refinement you would want to consider its labels suggestions, not even close to certaintly.

|                               | pH           | Temperature   |  Stirrate  | Other|
|------------------------------:|:------------:|:-------------:|:----------:|:----:|
|Short, Labeled                 | ✅           | ✅            |      ✅    |  ✅  |
|Short, Unlabeled               | ❌🤷         | ❌            |            |      |
|Large, Unlabeled               | ❌🤷         | ❌🤷          |            |      |
|Short, Unlabeled, More Context | ❌🤷         |               |      ✅    |      |
|Large, Unlabeled, More Context | ❌🤷         |               |            |      |
|Short, Unlabeled, Summary Stats|              | ❌🤷          |            |      |
|Short, Unlabeled, Less Context |              | ❌            |            |      |
|Short, Unlabeled, Units        |              | ✅🤷          |      ✅🤷  |  ✅  |
|Short, Unlabeled, Units Only   |              |               |      ✅    |  ✅🤷|
|Short, Labeled, Less Context   | ✅           | ✅            |      ✅    |  ✅  |

|Key: | ✅ Correctly Assessed | ❌ Incorrectly Assessed | 🤷 Below Confidence Gate | \[blank] Untested |
|----:|:---------------------:|:-----------------------:|:------------------------:|:----------------:|